# Data generation (instructor prep, not a student module)

This notebook builds the synthetic data shared by the modules:

1. **12 Groupe Mutuel reimbursement claim forms** (medical invoices) in `data/invoices/`, plus a **manually-authored ground truth** for each one, covering both:

   - the correct **extracted fields** (used to grade M02 — Structured Output), and
   - the correct **coverage decision** — approved / rejected / partial / insufficient information,
     the reimbursed amount, and a citation into the real `data/insurance_terms/*.pdf` documents
     (used to grade M03–M05).
2. **4 figures** in `data/insurance_figures/`, with questions whose answer is only in a figure (used by M03.04, visual RAG).

The two parts are independent: run only the one you need. Students never open this notebook; they only see its output.

**Why hand-authored instead of LLM-generated?** The spec asks for "Generated Ground-truth
validated manually" — hand-authoring the 12 scenarios directly against the real Groupe Mutuel
clauses (see below) gives an *exactly* correct, deterministic ground truth for free, which
matters because M05 grades against it live during the workshop. Each scenario cites a real
article we read directly out of the PDFs:

| Product / document | What it governs |
| --- | --- |
| `CAGA04-E2` — General Terms and Conditions (CGC) | Applies to every claim: general exclusions (Art. 18) — cosmetic treatment, dental unless a specific rider covers it, pre-existing conditions, self-inflicted harm, etc. |
| `DPAM01-E4` — Dentaire Plus | Dental care rider: reimbursement classes 1-5, each 75% up to a per-class annual cap; 1-month waiting period for non-accident treatment. |
| `SOGA01-E1` — Optimum | General supplemental rider: alternative medicine (75%, CHF 3,000/yr, CHF 500/yr sub-cap for massages), non-covered prescribed medication (90%), convalescence/thermal cures (90%, 30 days/yr). |
| `RFGM02-E3` (edition 2024-01-01) / `RFBA02-E5` (edition 2025-04-01) — PrimaCare | Compulsory-insurance GP-gatekeeper model. Two *editions* of the same product — the one in force on the treatment date applies. |

The 12 invoices deliberately include the failure modes the spec calls for: a capped claim, an
excluded category, a waiting-period rejection, a wrong-policy-document trap (two PrimaCare
editions), and an illegible/ambiguous field that a good extractor must flag rather than guess.

In [ ]:
import json
import platform
import random
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image, ImageDraw, ImageFilter, ImageFont

from agentic_workshop.config import DATA_DIR, INVOICES_DIR

GROUND_TRUTH_DIR = INVOICES_DIR / "ground_truth"
FIGURES_DIR = DATA_DIR / "insurance_figures"

_FONTS = {  # (regular, bold) per operating system
    "Darwin": (
        "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf",
    ),
    "Windows": ("C:/Windows/Fonts/arial.ttf", "C:/Windows/Fonts/arialbd.ttf"),
    "Linux": (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    ),
}


def font(size, bold=False):
    path = _FONTS.get(platform.system(), ("", ""))[bold]
    if path and Path(path).exists():
        return ImageFont.truetype(path, size)
    return ImageFont.load_default(size=size)  # Pillow >= 10.1


# Cross-platform fonts. The committed images were rendered with Arial on
# macOS: on another system the font, and so the pixels, differ slightly.
REGULAR = font(22)
BOLD = font(26, bold=True)
BOLD_SMALL = font(18, bold=True)

## Part 1: The claim forms

Rerunning this part overwrites `data/invoices/` and its ground truth.

### Render a claim form as an image

A plain `PIL`-drawn template, no external rendering dependency — deterministic and fast enough to regenerate live if a field needs tweaking. One invoice (`INV-2025-012`) gets a partially smudged amount field to simulate a hard-to-OCR scan: legible enough to tempt a guess, not legible enough to trust.

In [ ]:
def render_invoice(record, rng, smudge_amount=False):
    W, H = 900, 720
    img = Image.new("RGB", (W, H), "white")
    d = ImageDraw.Draw(img)
    d.rectangle([0, 0, W - 1, H - 1], outline=(200, 200, 200), width=2)
    d.text((40, 40), "Groupe Mutuel \u2013 Reimbursement Claim Form", font=BOLD, fill=(20, 20, 20))
    d.line([(40, 85), (W - 40, 85)], fill=(180, 180, 180), width=2)

    def field(label, value, y):
        d.text((40, y), label, font=BOLD_SMALL, fill=(60, 60, 60))
        d.text((320, y), str(value), font=REGULAR, fill=(0, 0, 0))
        return y + 38

    y = 110
    for label, key in [
        ("Invoice / Claim ID:", "invoice_id"),
        ("Patient name:", "patient_name"),
        ("Date of birth:", "patient_date_of_birth"),
        ("Policy holder no.:", "policy_holder_number"),
        ("Insurance product:", "insurance_product"),
        ("Policy start date:", "policy_start_date"),
    ]:
        y = field(label, record[key], y)
    y += 10
    d.line([(40, y), (W - 40, y)], fill=(180, 180, 180), width=1)
    y += 20
    for label, key in [
        ("Provider:", "provider_name"),
        ("Provider type:", "provider_type"),
        ("Treatment date:", "treatment_date"),
        ("Treatment description:", "treatment_description"),
        ("Notes:", "notes"),
    ]:
        y = field(label, record[key], y)
    y += 10
    d.line([(40, y), (W - 40, y)], fill=(180, 180, 180), width=1)
    y += 20
    d.text((40, y), "Total amount billed:", font=BOLD_SMALL, fill=(60, 60, 60))
    amount_str = f"CHF {record['amount_chf']:.2f}" if record["amount_chf"] is not None else "CHF ???.??"
    amount_pos = (320, y)
    d.text(amount_pos, amount_str, font=BOLD, fill=(0, 0, 0))

    if smudge_amount:
        bbox = d.textbbox(amount_pos, amount_str, font=BOLD)
        pad = 3
        mid_x = bbox[0] + int((bbox[2] - bbox[0]) * 0.55)
        box = (mid_x, bbox[1] - pad, bbox[2] + pad, bbox[3] + pad)
        region = img.crop(box).filter(ImageFilter.GaussianBlur(radius=2.2))
        img.paste(region, box)
        smudge = ImageDraw.Draw(img)
        for _ in range(18):
            x0 = rng.randint(box[0], box[2])
            y0 = rng.randint(box[1], box[3])
            smudge.ellipse([x0 - 1, y0 - 1, x0 + 1, y0 + 1], fill=(120, 120, 120))

    return img


### The 12 invoices

Each `dict` is exactly what appears on the rendered claim form. Patients recur across a few invoices (e.g. Léa Bernard has 4) so later modules can show per-patient annual caps accumulating across claims.

In [ ]:
RECORDS = [
    dict(
        invoice_id="INV-2025-001", patient_name="L\u00e9a Bernard", patient_date_of_birth="1990-05-02",
        policy_holder_number="GM-4471-2201", insurance_product="Dentaire Plus", policy_start_date="2020-03-01",
        provider_name="Cabinet Dentaire Rive Gauche", provider_type="Dentist", treatment_date="2025-03-10",
        treatment_description="Composite filling, tooth 26", notes="Routine care, illness", amount_chf=180.00,
    ),
    dict(
        invoice_id="INV-2025-002", patient_name="L\u00e9a Bernard", patient_date_of_birth="1990-05-02",
        policy_holder_number="GM-4471-2201", insurance_product="Dentaire Plus", policy_start_date="2020-03-01",
        provider_name="Cabinet Dentaire Rive Gauche", provider_type="Dentist", treatment_date="2025-09-15",
        treatment_description="Ceramic crown and bridge, teeth 14-16", notes="Routine care, illness", amount_chf=6000.00,
    ),
    dict(
        invoice_id="INV-2025-003", patient_name="L\u00e9a Bernard", patient_date_of_birth="1990-05-02",
        policy_holder_number="GM-4471-2201", insurance_product="Dentaire Plus", policy_start_date="2020-03-01",
        provider_name="Cabinet Dentaire Rive Gauche", provider_type="Dentist", treatment_date="2025-10-01",
        treatment_description="Cosmetic teeth whitening", notes="Elective, aesthetic", amount_chf=800.00,
    ),
    dict(
        invoice_id="INV-2025-004", patient_name="Marc Dubois", patient_date_of_birth="1985-11-20",
        policy_holder_number="GM-5820-3312", insurance_product="Optimum", policy_start_date="2019-01-01",
        provider_name="Cabinet d'Ost\u00e9opathie Lausanne", provider_type="Osteopath", treatment_date="2025-04-20",
        treatment_description="Osteopathy session, lower back pain", notes="Recognised alternative medicine practitioner", amount_chf=140.00,
    ),
    dict(
        invoice_id="INV-2025-005", patient_name="Marc Dubois", patient_date_of_birth="1985-11-20",
        policy_holder_number="GM-5820-3312", insurance_product="Optimum", policy_start_date="2019-01-01",
        provider_name="Institut de Massage Bien-\u00eatre", provider_type="Massage therapist", treatment_date="2025-04-25",
        treatment_description="Therapeutic massage, 60 min", notes="First massage claim this calendar year", amount_chf=90.00,
    ),
    dict(
        invoice_id="INV-2025-006", patient_name="Sofia Rossi", patient_date_of_birth="1978-02-14",
        policy_holder_number="GM-6193-4410", insurance_product="Optimum", policy_start_date="2015-06-01",
        provider_name="Clinique Thermale Val d'Anniviers", provider_type="Convalescence clinic", treatment_date="2025-06-01",
        treatment_description="Thermal cure, 12 days, prescribed by Dr. F. Moret (GP)", notes="Prescription attached", amount_chf=2400.00,
    ),
    dict(
        invoice_id="INV-2025-007", patient_name="Sofia Rossi", patient_date_of_birth="1978-02-14",
        policy_holder_number="GM-6193-4410", insurance_product="Optimum", policy_start_date="2015-06-01",
        provider_name="Pharmacie Centrale Sion", provider_type="Pharmacy", treatment_date="2025-06-10",
        treatment_description="Non-reimbursable prescribed medication (LPPA list)", notes="Prescribed by Dr. F. Moret", amount_chf=60.00,
    ),
    dict(
        invoice_id="INV-2025-008", patient_name="Ahmed Al-Farsi", patient_date_of_birth="1992-07-08",
        policy_holder_number="GM-7745-1183", insurance_product="Optimum", policy_start_date="2021-09-01",
        provider_name="Clinique de Chirurgie Esth\u00e9tique Gen\u00e8ve", provider_type="Plastic surgeon", treatment_date="2025-05-05",
        treatment_description="Rhinoplasty (aesthetic nose correction)", notes="Elective, not accident-related", amount_chf=4500.00,
    ),
    dict(
        invoice_id="INV-2025-009", patient_name="Ahmed Al-Farsi", patient_date_of_birth="1992-07-08",
        policy_holder_number="GM-7745-1183", insurance_product="Optimum", policy_start_date="2021-09-01",
        provider_name="Cabinet Dentaire Plainpalais", provider_type="Dentist", treatment_date="2025-07-12",
        treatment_description="Root canal treatment, tooth 36", notes="No Dentaire Plus rider on file", amount_chf=900.00,
    ),
    dict(
        invoice_id="INV-2025-010", patient_name="Peter Huber", patient_date_of_birth="1988-01-30",
        policy_holder_number="GM-8802-9021", insurance_product="Dentaire Plus", policy_start_date="2025-07-20",
        provider_name="Zahnarztpraxis Huber & Partner", provider_type="Dentist", treatment_date="2025-08-01",
        treatment_description="Composite filling, tooth 14", notes="Illness (cavity), not an accident", amount_chf=150.00,
    ),
    dict(
        invoice_id="INV-2025-011", patient_name="Nadia Keller", patient_date_of_birth="1995-09-12",
        policy_holder_number="GM-9012-5567", insurance_product="PrimaCare", policy_start_date="2023-01-01",
        provider_name="Walk-in Clinic Bahnhofstrasse", provider_type="General practitioner (not registered GP)", treatment_date="2025-05-15",
        treatment_description="Consultation for persistent cough", notes="Not seen by registered PrimaCare GP; not an emergency", amount_chf=220.00,
    ),
    dict(
        invoice_id="INV-2025-012", patient_name="L\u00e9a Bernard", patient_date_of_birth="1990-05-02",
        policy_holder_number="GM-4471-2201", insurance_product="Dentaire Plus", policy_start_date="2020-03-01",
        provider_name="Cabinet Dentaire Rive Gauche", provider_type="Dentist", treatment_date="2025-11-02",
        treatment_description="Emergency repair, broken tooth 11 (cycling fall)", notes="Accident: cycling fall", amount_chf=950.00,
    ),
]

print(f"{len(RECORDS)} invoice records defined")


### Ground truth: the correct coverage decision for each invoice

Hand-computed against the articles quoted above — this is what M05 grades the finished agent against.

In [ ]:
COVERAGE_GROUND_TRUTH = {
    "INV-2025-001": dict(verdict="approved", reimbursed_amount_chf=135.00,
        rationale="Class 1 dental filling, 75% of CHF 180 = CHF 135, under the CHF 1,000/year Class 1 cap.",
        citations=["DPAM01-E4 Art.3.1 (Class 1)"]),
    "INV-2025-002": dict(verdict="partial", reimbursed_amount_chf=4000.00,
        rationale="Class 3 crown/bridge work: 75% of CHF 6,000 = CHF 4,500, but Class 3 is capped at "
                   "CHF 4,000/calendar year, so reimbursement is capped at CHF 4,000.",
        citations=["DPAM01-E4 Art.3.1 (Class 3)"]),
    "INV-2025-003": dict(verdict="rejected", reimbursed_amount_chf=0.00,
        rationale="Cosmetic/aesthetic treatment (teeth whitening) is excluded under the general "
                   "conditions, regardless of the dental rider.",
        citations=["CAGA04-E2 Art.18.1.f"]),
    "INV-2025-004": dict(verdict="approved", reimbursed_amount_chf=105.00,
        rationale="Osteopathy is a recognised alternative medicine treatment: 75% of CHF 140 = CHF 105, "
                   "well under the CHF 3,000/year pool cap.",
        citations=["SOGA01-E1 Art.5 (Alternative medicine treatments)"]),
    "INV-2025-005": dict(verdict="approved", reimbursed_amount_chf=67.50,
        rationale="Massage is part of the alternative medicine pool but has its own CHF 500/year "
                   "sub-cap; 75% of CHF 90 = CHF 67.50, first massage claim this year, under sub-cap.",
        citations=["SOGA01-E1 Art.5 (Alternative medicine treatments \u2013 massages sub-cap)"]),
    "INV-2025-006": dict(verdict="approved", reimbursed_amount_chf=2160.00,
        rationale="Prescribed thermal cure, 12 days (under the 30 days/year cap): 90% of CHF 2,400 = CHF 2,160.",
        citations=["SOGA01-E1 Art.5 (Convalescence and thermal cures)"]),
    "INV-2025-007": dict(verdict="approved", reimbursed_amount_chf=54.00,
        rationale="Prescribed medication not covered by compulsory insurance: 90% of CHF 60 = CHF 54.",
        citations=["SOGA01-E1 Art.5 (Non-reimbursable drugs and restricted medication)"]),
    "INV-2025-008": dict(verdict="rejected", reimbursed_amount_chf=0.00,
        rationale="Elective aesthetic surgery not required following an insured event is explicitly excluded.",
        citations=["CAGA04-E2 Art.18.1.f"]),
    "INV-2025-009": dict(verdict="rejected", reimbursed_amount_chf=0.00,
        rationale="Dental treatment is excluded under the general conditions unless expressly covered "
                   "by a specific product; this patient has no Dentaire Plus rider.",
        citations=["CAGA04-E2 Art.18.1.c"]),
    "INV-2025-010": dict(verdict="rejected", reimbursed_amount_chf=0.00,
        rationale="Policy started 2025-07-20; non-accident dental benefits are subject to a one-month "
                   "waiting period, and this illness-related filling on 2025-08-01 falls within that window.",
        citations=["DPAM01-E4 Art.4.2"]),
    "INV-2025-011": dict(verdict="approved", reimbursed_amount_chf=None,
        rationale="Seeing a non-registered GP without meeting a listed exception is a compliance matter, "
                   "but per Art.7 a single instance only triggers a reminder; it does not reduce "
                   "reimbursement for this specific claim (exclusion from the model only follows two "
                   "reminders). The claim should be processed at the standard compulsory-insurance rate. "
                   "Note the applicable edition is RFBA02-E5 (effective 2025-04-01), not the older "
                   "RFGM02-E3 (effective 2024-01-01), since the treatment date (2025-05-15) postdates "
                   "the newer edition.",
        citations=["RFBA02-E5 Art.7"]),
    "INV-2025-012": dict(verdict="insufficient_information", reimbursed_amount_chf=None,
        rationale="The billed amount is smudged/illegible on the form (reads as CHF 9??.??); extraction "
                   "should flag this field as low-confidence rather than guessing, and the claim should "
                   "be sent back for a legible copy before a coverage decision can be made.",
        citations=[]),
}


### Render, save, and write ground truth files

In [ ]:
rng = random.Random(42)  # deterministic smudge pattern
GROUND_TRUTH_DIR.mkdir(parents=True, exist_ok=True)

for rec in RECORDS:
    smudge = rec["invoice_id"] == "INV-2025-012"
    img = render_invoice(rec, rng, smudge_amount=smudge)
    img.save(INVOICES_DIR / f"{rec['invoice_id']}.png")

    extraction_gt = dict(rec)
    if smudge:
        extraction_gt["amount_chf"] = None
        extraction_gt["extraction_note"] = (
            "Amount is smudged/illegible on the source document "
            "(plausible reads: 900.00 or 950.00)."
        )

    full_gt = dict(
        invoice_id=rec["invoice_id"],
        extraction=extraction_gt,
        coverage_decision=COVERAGE_GROUND_TRUTH[rec["invoice_id"]],
    )
    (GROUND_TRUTH_DIR / f"{rec['invoice_id']}.json").write_text(
        json.dumps(full_gt, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Wrote {len(RECORDS)} images to {INVOICES_DIR}")
print(f"Wrote {len(RECORDS)} ground truth files to {GROUND_TRUTH_DIR}")

### Sanity check: preview one normal invoice and the smudged one

In [ ]:
display(Image.open(INVOICES_DIR / "INV-2025-001.png"))
display(Image.open(INVOICES_DIR / "INV-2025-012.png"))

## Part 2: The figures (for M03.04, visual RAG)

Four charts of **invented** statistics, under Groupe Mutuel's product names. The policy PDFs in `data/insurance_terms/` never contain these numbers, so a text-only RAG cannot answer the questions below: the answer is only in a figure. Because we generate the charts, the ground truth is exact.

Each figure is square-ish (6 × 5.6 in): CLIP, used in M03.04's optional extension, centre-crops images to a square.

In [ ]:
YEARS = list(range(2018, 2027))
PREMIUMS = {  # monthly premium, CHF
    "Standard (free choice)": [380, 395, 410, 428, 445, 463, 482, 501, 520],
    "PrimaCare": [289, 301, 312, 318, 324, 329, 333, 336, 338],
    "OptiMed":   [245, 262, 281, 299, 316, 341, 366, 392, 418],
}
DENTAL_BY_CANTON = {"Geneva": 412, "Basel-Stadt": 402, "Vaud": 388, "Zurich": 371, "Ticino": 355,
                    "Bern": 341, "Valais": 329, "Fribourg": 318, "Lucerne": 306}  # average claim, CHF
COST_SPLIT = {"Outpatient care": 38, "Hospital": 27, "Medication": 19, "Other benefits": 6,
              "Administration": 6, "Prevention": 4}  # % of premium income
SETTLEMENT_COUNTS = [180, 1240, 2960, 1870, 820, 410, 190, 70]  # claims per 5-working-day bin: 0-5, 5-10, ...


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / name)
    plt.close(fig)


FIGURES_DIR.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 5.6), dpi=100)
for (label, values), marker in zip(PREMIUMS.items(), ["^", "o", "s"], strict=True):
    ax.plot(YEARS, values, marker=marker, linewidth=2.5, markersize=6, label=label)
ax.set_title("Average monthly premium by model, 2018–2026", fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Year"); ax.set_ylabel("Monthly premium (CHF)")
ax.legend(fontsize=10, loc="upper left"); ax.grid(alpha=0.3)
save(fig, "fig_premiums.png")

fig, ax = plt.subplots(figsize=(6, 5.6), dpi=100)
names = list(DENTAL_BY_CANTON)[::-1]
ax.barh(names, [DENTAL_BY_CANTON[n] for n in names], color="#2a9d8f")
ax.set_title("Average Dentaire Plus claim by canton, 2025", fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Average claim (CHF)")
for i, n in enumerate(names):
    ax.text(DENTAL_BY_CANTON[n] + 5, i, str(DENTAL_BY_CANTON[n]), va="center", fontsize=9)
ax.set_xlim(0, 470)
save(fig, "fig_dental_by_canton.png")

fig, ax = plt.subplots(figsize=(6, 5.6), dpi=100)
ax.pie(list(COST_SPLIT.values()), labels=list(COST_SPLIT), autopct="%1.0f%%", startangle=90,
       wedgeprops={"width": 0.42}, textprops={"fontsize": 10})
ax.set_title("Where your premium franc goes, 2025", fontsize=14, fontweight="bold", pad=14)
save(fig, "fig_cost_split.png")

fig, ax = plt.subplots(figsize=(6, 5.6), dpi=100)
ax.bar([2.5 + 5 * i for i in range(len(SETTLEMENT_COUNTS))], SETTLEMENT_COUNTS, width=4.6, color="#e76f51", edgecolor="white")
ax.set_title("Working days from claim submission to reimbursement", fontsize=13, fontweight="bold", pad=14)
ax.set_xlabel("Working days"); ax.set_ylabel("Number of claims")
ax.grid(axis="y", alpha=0.3)
save(fig, "fig_claims_processing.png")

print("\n".join(p.name for p in sorted(FIGURES_DIR.glob("*.png"))))

### Ground truth: one question per figure

Computed from the data behind each chart. `expected` lists strings a correct answer must contain (used by M03.04 to check the answers).

In [ ]:
primacare, optimed = PREMIUMS["PrimaCare"], PREMIUMS["OptiMed"]
crossover = next(year for year, p, o in zip(YEARS, primacare, optimed, strict=True) if o > p)
top_canton = max(DENTAL_BY_CANTON, key=DENTAL_BY_CANTON.get)
busiest_bin = max(range(len(SETTLEMENT_COUNTS)), key=SETTLEMENT_COUNTS.__getitem__)

FIGURE_QUESTIONS = [
    dict(figure="fig_premiums.png",
         question="In which year did the average OptiMed premium first rise above the average PrimaCare premium?",
         answer=f"{crossover} (OptiMed CHF {optimed[YEARS.index(crossover)]}, PrimaCare CHF {primacare[YEARS.index(crossover)]})",
         expected=[str(crossover)]),
    dict(figure="fig_dental_by_canton.png",
         question="Which canton has the highest average Dentaire Plus claim, and how much is it?",
         answer=f"{top_canton}, CHF {DENTAL_BY_CANTON[top_canton]}",
         expected=[top_canton, str(DENTAL_BY_CANTON[top_canton])]),
    dict(figure="fig_cost_split.png",
         question="What share of premium income goes to administration?",
         answer=f"{COST_SPLIT['Administration']}%",
         expected=[f"{COST_SPLIT['Administration']}%"]),
    dict(figure="fig_claims_processing.png",
         question="What is the most common time from claim submission to reimbursement?",
         answer=f"{5 * busiest_bin}–{5 * busiest_bin + 5} working days",
         expected=[str(5 * busiest_bin), str(5 * busiest_bin + 5)]),
]

(FIGURES_DIR / "ground_truth.json").write_text(json.dumps(FIGURE_QUESTIONS, indent=2, ensure_ascii=False), encoding="utf-8")
for q in FIGURE_QUESTIONS:
    print(f"{q['figure']:28s} {q['answer']}")

In [ ]:
display(Image.open(FIGURES_DIR / "fig_premiums.png"))